# 10.4 · ResNet 与跳跃连接 / ResNet & Skip Connections

> **课程定位 / Where this fits**
> 第 4 课，**Part 10 · 计算机视觉**。
> Lesson 4, **Part 10 · Computer Vision**.
>
> VGG 告诉我们"更深更好"，但人们很快发现：**朴素(plain)网络堆得太深，反而更差**——而且不是过拟合，是**连训练精度都更低**！这叫**退化问题(degradation)**。2015 年 **ResNet** 用一个惊人简单的"捷径"——**残差连接(skip connection)**——解决了它，让网络能堆到 100、1000 层，夺得 ImageNet 冠军、超越人类。本课**复现退化**、搭建残差块、并用**梯度流可视化**揭示残差为何有效。
> VGG said "deeper is better," but people found that **stacking plain nets too deep makes them worse** — not from overfitting; even **training accuracy drops**! This is the **degradation problem**. In 2015 **ResNet** fixed it with a stunningly simple shortcut — the **residual/skip connection** — enabling 100-/1000-layer nets, winning ImageNet, beating humans. We'll **reproduce degradation**, build a residual block, and reveal why it works via **gradient-flow visualization**.
>
> 💼 **实战/面试视角**："残差连接为什么有效 / 退化问题 / 恒等映射" 是深度学习面试 Top 题。
> 💼 **Practical/interview angle:** "why residuals work / degradation / identity mapping" — top deep-learning interview questions.

> 📐 **符号约定 / Notation**
> - $x$ 输入, $H(x)$ 期望输出, $F(x)=H(x)-x$ 残差 / input, target, residual
> - skip/shortcut —— 把输入直接加到输出的"捷径" / the shortcut adding input to output

> 💡 **面试相关 / Interview-relevant**
> - "残差连接为什么能训练很深的网络"（出镜率 ★★★★★）
> - "退化问题是什么(不是过拟合)"（★★★★）
> - "为什么学残差 F(x)=H(x)−x 比直接学 H(x) 容易"（★★★★）
> - "残差如何缓解梯度消失"（★★★★，梯度高速公路）

---

## 学习目标 / Learning Objectives
1. 理解并**复现退化问题**（深 plain 网络精度更低）。
   Understand and reproduce the degradation problem.
2. 掌握**残差块**结构与"学残差"的思想。
   Master the residual block and the "learn the residual" idea.
3. 验证残差网络能正常加深、优于同深度 plain。
   Verify residual nets train deep and beat same-depth plain nets.
4. 用**梯度流可视化**解释残差为何有效（梯度高速公路）。
   Explain residuals via gradient-flow visualization (gradient highway).

## 目录 / TOC
1. [退化问题：深 ≠ 好（复现）⭐](#1)
2. [残差块：一条捷径 ⭐](#2)
3. [残差 vs Plain：同深度对比 ⭐](#3)
4. [为什么有效：梯度流 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 退化问题：深 ≠ 好（复现）⭐ / The Degradation Problem

直觉上，更深的网络至少不该比浅的差——多出来的层"什么都不做(学成恒等映射)"就行了。但 ResNet 论文发现：**朴素地堆深，网络反而更难优化**——经典证据是 56 层 plain 网络的**训练误差**竟比 20 层的更高。这叫**退化(degradation)**，它**不是过拟合**（过拟合是训练好测试差），而是个**优化难题**——深层网络梯度难以有效传播（呼应 9.9 的梯度消失，§4 会亲眼验证）。
Intuitively a deeper net shouldn't be worse — extra layers could just "do nothing (identity)." But the ResNet paper found **naively going deeper hurts optimization** — classically, a 56-layer plain net had *higher training error* than a 20-layer one. This **degradation is not overfitting** (which is good-train/bad-test); it's an *optimization* problem — gradients struggle through deep nets (echoing 9.9, verified in §4).

下面用一个**快速小实验**感受："加深 plain 网络并没有带来更好的结果"。两者都用 BatchNorm（现代标配，否则深网根本训不动），唯一差别是深度。
A **quick small experiment** to feel it: "deepening a plain net brings no gain." Both use BatchNorm (modern standard; without it deep nets don't train at all), differing only in depth.

> ⚠️ 受限于 CPU 与训练时长，这个小实验只能展示"加深没好处"（测试精度不升反降）；要复现原论文那种"训练误差随深度上升"的经典退化，需要更极端的深度。**最干净的证据在 §3（同深度对照）和 §4（梯度流）**。
> ⚠️ Given CPU/time limits, this small demo only shows "depth gives no benefit" (test accuracy stalls/drops); reproducing the paper's "training error rises with depth" needs far greater depth. **The cleanest evidence is in §3 (same-depth) and §4 (gradient flow).**


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
sns.set_theme(style="whitegrid")

DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")
tfm = transforms.ToTensor()
train_full = torchvision.datasets.FashionMNIST(DATA_ROOT, train=True, download=True, transform=tfm)
test_full  = torchvision.datasets.FashionMNIST(DATA_ROOT, train=False, download=True, transform=tfm)
train_loader = DataLoader(Subset(train_full, range(5000)), batch_size=128, shuffle=True)
test_loader  = DataLoader(Subset(test_full, range(2000)), batch_size=256)

class PlainBlock(nn.Module):                              # 朴素块(含BN), 无捷径 / plain block with BN, no shortcut
    def __init__(self, c):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(c,c,3,padding=1), nn.BatchNorm2d(c), nn.ReLU(),
                                  nn.Conv2d(c,c,3,padding=1), nn.BatchNorm2d(c))
        self.relu = nn.ReLU()
    def forward(self, x): return self.relu(self.conv(x))

def build(block, n_blocks, c=16):
    layers = [nn.Conv2d(1, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU()]
    for _ in range(n_blocks): layers.append(block(c))     # 堆叠 n 个块 / stack n blocks
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, 10)]
    return nn.Sequential(*layers)

def train_eval(model, epochs=3):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); ce = nn.CrossEntropyLoss(); hist=[]
    for _ in range(epochs):
        model.train(); ep=[]
        for xb, yb in train_loader:
            opt.zero_grad(); loss = ce(model(xb), yb); loss.backward(); opt.step(); ep.append(loss.item())
        hist.append(np.mean(ep))
    model.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            correct += (model(xb).argmax(1)==yb).sum().item(); total += len(yb)
    return hist, correct/total

depths = {"浅 plain (3块~8层)": 3, "深 plain (8块~18层)": 8}
plain_results = {}
for name, n in depths.items():
    torch.manual_seed(0); plain_results[name] = train_eval(build(PlainBlock, n))
fig, ax = plt.subplots(figsize=(7,4))
for name, (hist, acc) in plain_results.items():
    ax.plot(hist, "o-", label=f"{name}  acc={acc:.3f}")
ax.set_xlabel("epoch"); ax.set_ylabel("训练损失"); ax.legend()
ax.set_title("退化迹象: 朴素加深, 测试精度不升反降(深度没转化为更好结果)")
plt.tight_layout(); plt.show()
for name, (hist, acc) in plain_results.items():
    print(f"{name}: 最终训练损失={hist[-1]:.3f}, test 准确率={acc:.3f}")
print("→ 加深 plain 网络后测试精度没有提升(甚至下降): 深度没带来好处")
print("(经典退化是'训练误差随深度上升', 需更极端深度才明显; 干净证据见 §3 同深度对照 + §4 梯度流)")


<a id="2"></a>
## 2. 残差块：一条捷径 ⭐ / Residual Block: A Shortcut

ResNet 的想法极其简单却深刻。普通块直接学习目标映射 $H(x)$；残差块改成：让卷积层只学**残差** $F(x) = H(x) - x$，最终输出 $F(x) + x$——即把**输入 $x$ 通过一条"捷径"直接加到输出**。
ResNet's idea is simple yet profound. A plain block learns the target $H(x)$ directly; a residual block instead learns only the **residual** $F(x) = H(x) - x$, output $F(x) + x$ — i.e. **add the input $x$ to the output via a shortcut**.

**为什么这样更容易学**（面试核心）：如果某层的最优解就是"什么都不改(恒等映射)"，普通块要费劲让一堆卷积逼近恒等；而残差块只需把 $F(x)$ 学成 **0**（权重趋零即可）——**学 0 比学恒等容易得多**。这保证"加深至少不会变差"。
**Why easier to learn** (interview core): if the optimal is "change nothing (identity)," a plain block struggles to make convs approximate identity; a residual block just needs $F(x)=0$ (weights→0) — **learning 0 is far easier than learning identity**. This guarantees "deeper won't hurt."


In [ ]:
class ResBlock(nn.Module):                               # 残差块(含BN): 输出 = F(x) + x / residual block
    def __init__(self, c):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(c,c,3,padding=1), nn.BatchNorm2d(c), nn.ReLU(),
                                  nn.Conv2d(c,c,3,padding=1), nn.BatchNorm2d(c))
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.conv(x) + x)               # 关键: 加回输入 x (跳跃连接) / add input back

# 演示"恒等友好": 把残差分支权重全置0 → F(x)=0 → 残差块输出=relu(x), 几乎是恒等 / zero F(x) → identity
demo = nn.Sequential(nn.Conv2d(4,4,3,padding=1), nn.Conv2d(4,4,3,padding=1))  # 简化残差分支(无BN便于演示)
for p in demo.parameters(): nn.init.zeros_(p)            # F(x) 权重全置0 / make F(x)=0
x = torch.randn(1, 4, 8, 8)
out = torch.relu(demo(x) + x)                            # = relu(0 + x) = relu(x)
print(f"残差分支 F(x)=0 时, 残差块输出 = relu(x); (F(x)+x 与 x 的差异 = {(demo(x)+x - x).abs().max().item():.4f})")
print("→ 残差块'天生'容易表示恒等映射: 多加的层最差也就'什么都不做', 不会拖累网络 → 解决退化")


<a id="3"></a>
## 3. 残差 vs Plain：同深度对比 ⭐ / Residual vs Plain at Same Depth

把 §1 中那个"退化"的**深 plain 网络**，**仅仅把 PlainBlock 换成 ResBlock**（其它完全一样），看精度是否恢复甚至更好。这是验证残差威力的关键对照实验。
Take the "degraded" **deep plain net** from §1 and **only swap PlainBlock for ResBlock** (everything else identical). See if accuracy recovers or improves. The key controlled experiment.


In [ ]:
deep_plain_hist, deep_plain_acc = plain_results["深 plain (8块~18层)"]      # 复用 §1 结果 / reuse §1
torch.manual_seed(0)
deep_res_hist, deep_res_acc = train_eval(build(ResBlock, 8))                 # 同深度, 改残差 / same depth, residual
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(deep_plain_hist, "s-", label=f"深 plain  acc={deep_plain_acc:.3f}", color="#e67")
axes[0].plot(deep_res_hist, "o-", label=f"深 residual  acc={deep_res_acc:.3f}", color="#39c")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("训练损失"); axes[0].legend(); axes[0].set_title("同深度: 加残差 → 损失更低更快")
axes[1].bar(["深 plain","深 residual"], [deep_plain_acc, deep_res_acc], color=["#e67","#39c"])
for i,v in enumerate([deep_plain_acc, deep_res_acc]): axes[1].text(i, v+0.005, f"{v:.3f}", ha="center")
axes[1].set_ylabel("test 准确率"); axes[1].set_title("仅加跳跃连接, 精度大幅提升")
plt.tight_layout(); plt.show()
print(f"深 plain    : test 准确率 = {deep_plain_acc:.3f}")
print(f"深 residual : test 准确率 = {deep_res_acc:.3f}  ← 仅加跳跃连接就解决退化")
print("结论: 残差连接让深网络重新可优化; 这就是 ResNet 能堆到 100+ 层的原因")


<a id="4"></a>
## 4. 为什么有效：梯度流 + 小结 ⭐ / Why It Works: Gradient Flow

残差为何能救深网络？最直接的解释是**梯度流**。反向传播时，残差块的 $+x$ 捷径让梯度多一条**不经过卷积层、直接传回浅层的通道**（导数里多了个"$+1$"）。即使很深，浅层也能收到健康梯度，不会消失。
Why do residuals rescue deep nets? The most direct explanation is **gradient flow**. In backprop, the $+x$ shortcut gives gradients a path **straight back to early layers without passing through convs** (a "$+1$" term in the derivative). Even when very deep, early layers still get healthy gradients.

为了**干净地隔离残差的作用**，这里特意**不用 BatchNorm**（BN 自己也能稳梯度），让 plain 与 residual 的唯一区别就是那条捷径。我们测每个卷积层权重的**梯度范数**：plain 越往浅层梯度越小（消失），residual 保持平稳。
To **cleanly isolate the residual's effect**, here we deliberately **drop BatchNorm** (BN also stabilizes gradients), so the only difference is the shortcut. We measure each conv layer's **gradient norm**: plain shrinks toward early layers (vanishing), residual stays steady.


In [ ]:
# 为隔离"跳跃连接"的作用, 这两个块都不含 BN / no-BN blocks to isolate the skip's effect
class PlainNB(nn.Module):
    def __init__(self, c):
        super().__init__(); self.conv=nn.Sequential(nn.Conv2d(c,c,3,padding=1),nn.ReLU(),nn.Conv2d(c,c,3,padding=1)); self.relu=nn.ReLU()
    def forward(self, x): return self.relu(self.conv(x))
class ResNB(nn.Module):
    def __init__(self, c):
        super().__init__(); self.conv=nn.Sequential(nn.Conv2d(c,c,3,padding=1),nn.ReLU(),nn.Conv2d(c,c,3,padding=1)); self.relu=nn.ReLU()
    def forward(self, x): return self.relu(self.conv(x) + x)        # 唯一区别: +x / only difference: +x
def build_nb(block, n, c=16):
    L=[nn.Conv2d(1,c,3,padding=1), nn.ReLU()]
    for _ in range(n): L.append(block(c))
    L+=[nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c,10)]; return nn.Sequential(*L)

def layer_grad_norms(model):
    """反传一次, 收集每个卷积层权重的梯度范数 / one backward, per-conv grad norms."""
    ce = nn.CrossEntropyLoss(); xb, yb = next(iter(train_loader))
    model.zero_grad(); ce(model(xb), yb).backward()
    return [m.weight.grad.norm().item() for m in model.modules() if isinstance(m, nn.Conv2d)]

torch.manual_seed(0); gp = layer_grad_norms(build_nb(PlainNB, 12))    # 深(12块=26层)无BN / deep no-BN
torch.manual_seed(0); gr = layer_grad_norms(build_nb(ResNB, 12))
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(range(len(gp)), gp, "s-", label="plain (梯度往浅层急剧衰减)", color="#e67")
ax.plot(range(len(gr)), gr, "o-", label="residual (梯度全程健康)", color="#39c")
ax.set_yscale("log"); ax.set_xlabel("卷积层(0=最浅 → 最深)"); ax.set_ylabel("梯度范数(对数轴)")
ax.legend(); ax.set_title("梯度流: 残差的'捷径'给梯度开了高速公路, 浅层也收到健康梯度")
plt.tight_layout(); plt.show()
print(f"plain    最浅层梯度范数 = {gp[0]:.2e}  (几乎为0 → 梯度消失, 浅层学不动)")
print(f"residual 最浅层梯度范数 = {gr[0]:.2e}  (健康, 大了很多个数量级)")
print("残差导数里多一个'+1'项 → 梯度能绕过卷积直接回传浅层 → 从根上缓解梯度消失")


```
退化问题: 朴素堆深→训练精度反而更低(优化难题, 不是过拟合); 主因深层梯度难传播
残差块: 输出=F(x)+x; 卷积层只学残差 F(x)=H(x)−x; 一条捷径把输入加到输出
为什么易学: 若最优是恒等, 残差只需 F(x)=0(学0比学恒等易) → 加深至少不变差
梯度流: +x 捷径让梯度多一个'+1'项, 绕过卷积直接回浅层 → 解决梯度消失
效果: 仅加跳跃连接, 同深度网络从退化恢复; ResNet 由此能堆 100~1000 层
配套: 现代深网络几乎都配 BatchNorm(否则深网根本训不动); bottleneck 用1×1降算力
```

### 💡 面试速查 / Interview cheat-sheet
1. **退化问题**: 深 plain 网络连训练都更差(优化难, 非过拟合)。
   Degradation: deep plain nets are worse even on training (optimization, not overfitting).
2. **残差块**: 输出 F(x)+x, 只学残差; 学恒等变容易。
   Residual block: output F(x)+x, learn the residual; identity becomes easy.
3. **为什么有效**: 恒等友好 + 梯度高速公路(导数+1项)。
   Why: identity-friendly + gradient highway (the "+1" term).
4. **效果**: 能训练 100~1000 层, 2015 ImageNet 夺冠超人类。
   Effect: trains 100–1000 layers; won 2015 ImageNet, beat humans.
5. **bottleneck**: 1×1降维-3×3-1×1升维, 省算力(深层 ResNet 用)。
   Bottleneck: 1×1 reduce-3×3-1×1 expand (deep ResNets).

### 下一节 / Next
**10.5 数据增强**——再好的架构也需要足够数据。数据增强通过翻转/裁剪/Mixup/Cutout 等"免费"扩充训练集，是提升泛化、对抗过拟合的实战利器。我们会**可视化各种增强**并实测它带来的精度提升。
**10.5 Data Augmentation** — even great architectures need enough data. Augmentation "freely" expands the training set via flip/crop/Mixup/Cutout, a practical tool for generalization. We'll **visualize augmentations** and measure the accuracy gain.
